# 02 — Preprocessing
Clean the data, select features, encode labels, scale, and split into train/test sets.

In [1]:
# Cell 1 — Imports
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
# Cell 2 — Load primary training data
df = pd.read_csv("../data/SpotifySongs.csv")
print(f"Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

Loaded: 114,000 rows, 21 columns


In [3]:
# Cell 3 — Clean: drop duplicates and null rows
before = len(df)

df = df.drop_duplicates()
dupes_removed = before - len(df)

df = df.dropna()
nulls_removed = before - dupes_removed - len(df)

print(f"Duplicates removed : {dupes_removed:,}")
print(f"Null rows removed  : {nulls_removed:,}")
print(f"Rows remaining     : {len(df):,}")

Duplicates removed : 0
Null rows removed  : 1
Rows remaining     : 113,999


In [4]:
# Cell 4 — Feature selection: keep only audio features + target
AUDIO_FEATURES = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
]
TARGET = 'track_genre'

df = df[AUDIO_FEATURES + [TARGET]]
print(f"Columns kept: {df.columns.tolist()}")
df.head()

Columns kept: ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'track_genre']


,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,track_genre
0,0.676,0.4610,-6.746,0.1430,0.0322,0.000001,0.3580,0.715,87.917,acoustic
1,0.420,0.1660,-17.235,0.0763,0.9240,0.000006,0.1010,0.267,77.489,acoustic
2,0.438,0.3590,-9.734,0.0557,0.2100,0.000000,0.1170,0.120,76.332,acoustic
3,0.266,0.0596,-18.515,0.0363,0.9050,0.000071,0.1320,0.143,181.740,acoustic
4,0.618,0.4430,-9.681,0.0526,0.4690,0.000000,0.0829,0.167,119.949,acoustic


In [5]:
# Cell 5 — Encode target labels
le = LabelEncoder()
y = le.fit_transform(df[TARGET])

# Show the integer → genre mapping
print("Label encoding map:")
for i, genre in enumerate(le.classes_):
    print(f"  {i:>3} → {genre}")

Label encoding map:
    0 → acoustic
    1 → afrobeat
    2 → alt-rock
    3 → alternative
    4 → ambient
    5 → anime
    6 → black-metal
    7 → bluegrass
    8 → blues
    9 → brazil
   10 → breakbeat
   11 → british
   12 → cantopop
   13 → chicago-house
   14 → children
   15 → chill
   16 → classical
   17 → club
   18 → comedy
   19 → country
   20 → dance
   21 → dancehall
   22 → death-metal
   23 → deep-house
   24 → detroit-techno
   25 → disco
   26 → disney
   27 → drum-and-bass
   28 → dub
   29 → dubstep
   30 → edm
   31 → electro
   32 → electronic
   33 → emo
   34 → folk
   35 → forro
   36 → french
   37 → funk
   38 → garage
   39 → german
   40 → gospel
   41 → goth
   42 → grindcore
   43 → groove
   44 → grunge
   45 → guitar
   46 → happy
   47 → hard-rock
   48 → hardcore
   49 → hardstyle
   50 → heavy-metal
   51 → hip-hop
   52 → honky-tonk
   53 → house
   54 → idm
   55 → indian
   56 → indie
   57 → indie-pop
   58 → industrial
   59 → iranian
   60 → 

In [6]:
# Cell 6 — Scale features (fit on train, transform both splits)
X = df[AUDIO_FEATURES].values

# Train/test split first so the scaler is fit only on training data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")

X_train : (91199, 9)
X_test  : (22800, 9)


In [7]:
# Cell 7 — Verify class distribution in each split
train_counts = pd.Series(y_train).value_counts().sort_index()
test_counts  = pd.Series(y_test).value_counts().sort_index()

dist = pd.DataFrame({
    'genre'      : le.classes_,
    'train_count': train_counts.values,
    'test_count' : test_counts.values
})
print(dist.to_string(index=False))

            genre  train_count  test_count
         acoustic          800         200
         afrobeat          800         200
         alt-rock          800         200
      alternative          800         200
          ambient          800         200
            anime          800         200
      black-metal          800         200
        bluegrass          800         200
            blues          800         200
           brazil          800         200
        breakbeat          800         200
          british          800         200
         cantopop          800         200
    chicago-house          800         200
         children          800         200
            chill          800         200
        classical          800         200
             club          800         200
           comedy          800         200
          country          800         200
            dance          800         200
        dancehall          800         200
      death

In [8]:
# Cell 8 — Save processed arrays, scaler, and encoder
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

np.save("../data/processed/X_train.npy", X_train)
np.save("../data/processed/X_test.npy",  X_test)
np.save("../data/processed/y_train.npy", y_train)
np.save("../data/processed/y_test.npy",  y_test)

joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(le,     "../models/label_encoder.pkl")

print("Saved:")
print("  data/processed/X_train.npy")
print("  data/processed/X_test.npy")
print("  data/processed/y_train.npy")
print("  data/processed/y_test.npy")
print("  models/scaler.pkl")
print("  models/label_encoder.pkl")

Saved:
  data/processed/X_train.npy
  data/processed/X_test.npy
  data/processed/y_train.npy
  data/processed/y_test.npy
  models/scaler.pkl
  models/label_encoder.pkl
